In [298]:
# Necessary Libraries
import numpy as np              # For numeric operations
import pandas as pd             # For data manipulation and analysis
import matplotlib.pyplot as plt # For data visualization
%matplotlib inline              

# To visualize text
from wordcloud import WordCloud

# For Naturla Language Processing
import nltk
from nltk.corpus import stopwords

# Downloading NLTK data
nltk.download('stopwords') # Downloading stopwords data
nltk.download('punkt')     # Downloading tokenizer data

[nltk_data] Downloading package stopwords to C:\Users\Gangotri
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Gangotri
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [299]:
# Read CSV file
df = pd.read_csv('spam.csv')

# Display the first few rows of DataFrame
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [300]:
# Removing unnecessary columns
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'], inplace= True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [301]:
# Columns name change
df.rename(columns={'v1': 'target', 'v2': 'text'}, inplace= True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# Data Pre-processing

In [302]:
from sklearn.preprocessing import LabelEncoder

# Encoding text to numerical
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])

df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# Check duplicate values
df.duplicated().sum()

# Drop duplicate
df = df.drop_duplicates(keep='first')
len(df)

5169

# Feature Engineering

In [304]:
# Importing the Porter stemmer for text Stemming
from nltk.stem.porter import PorterStemmer

# Importing the string module for handling special characters
import string

# Creating an instance of the Porter Stemmer
ps = PorterStemmer()

punkt
Ye actual tokenizer model hota hai
Pre-trained statistical model
Sentence & word boundaries samjhta hai

punkt_tab
Ye supporting data / lookup tables hain
Language-specific rules ka database
Tokenizer ko batata hai:
abbreviations (Mr., Dr.)
punctuation behavior
sentence ending rules

new tokenizer doesn't work without 'punkt_tab'

In [305]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

# Lowercase transformation and text preprocessing function
def transform_text(text):
    # Transform text to lowercase
    text = text.lower()
    
    # Tokenization using NLTK
    text = nltk.word_tokenize(text)
    
    # Removing special characters
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)
    
    # Loop through the tokens and remove stopwords and punctuation
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    # Stemming using Porter Stemmer
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    
    # Join the processed tokens back into a single string
    return " ".join(y)

[nltk_data] Downloading package punkt to C:\Users\Gangotri
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Gangotri
[nltk_data]     Mishra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [306]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazi avail onli in bugi...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni ok lar ... joke wif u on...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri in 2 a wkli comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so earli hor u c alreadi then say u ...
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i do think he goe to usf he live around he...


🔹 1. CountVectorizer
👉 Kya karta hai?

Har word kitni baar aaya → count karta hai

Output = word frequency matrix


🔹 2. TF-IDF Vectorizer
👉 Kya karta hai?

TF (Term Frequency) → word kitni baar document me aaya

IDF (Inverse Document Frequency) → word kitne documents me aaya

👉 Rare words = higher weight
👉 Common words = lower weight

In [307]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Help us convert text to vector | very naive
cv = CountVectorizer(max_features=500)

# Help us convert text to vector | Advance
tv = TfidfVectorizer(max_features = 500)

🔹 scipy.sparse._csr.csr_matrix kya hai? why we have to convert tv.fit_transform(df['transformed_text']) to array

👉 Ye SciPy ka sparse matrix data structure hai
👉 Specifically CSR = Compressed Sparse Row matrix
Ye memory-efficient way hai badi matrices store karne ka jahan zyada values 0 hoti hain, it stores mainly non-zero value and their position.

now we have to convert it in array to use it

In [308]:
X = tv.fit_transform(df['transformed_text']).toarray()
Y = df['target'].values

# Train Test Split

In [309]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.20, random_state = 2)

In [310]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [311]:
lrc = LogisticRegression(solver= 'liblinear', penalty= 'l1')
svc = SVC(kernel= "sigmoid", gamma= 1.0)
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth= 5)
knc = KNeighborsClassifier()
rfc = RandomForestClassifier(n_estimators= 50, random_state= 2)
abc = AdaBoostClassifier(n_estimators= 50, random_state= 2)
bgc  = BaggingClassifier(n_estimators= 50, random_state= 2)
etc = ExtraTreesClassifier(n_estimators= 50, random_state= 2)
gbc = GradientBoostingClassifier(n_estimators= 50, random_state= 2)
xgb = XGBClassifier(n_estimator= 50, random_state= 2)

In [312]:
clfs = {
'lrc' : lrc,
'svc' : svc,
'mnb' : mnb,
'dtc' : dtc,
'knc' : knc,
'rfc' : rfc,
'abc' : abc,
'bgc' : bgc,
'etc' : etc,
'gbc' : gbc,
'xgb' : xgb
}

In [313]:
from sklearn.metrics import accuracy_score, precision_score

def train_classifier(clfs, x_train, x_test, y_train, y_test):
    clfs.fit(x_train, y_train)
    y_pred = clfs.predict(x_test)
    print(type(y_pred), type(y_test))
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    return accuracy, precision

In [314]:
accuracy_list = []
precision_list = []
clfs_result = dict()
for name, clfs in clfs.items():
    acc, prec = train_classifier(clfs,  x_train, x_test, y_train, y_test)
    
    print("Classifier: ", name)
    print("Accuracy: ", acc)
    print("Precision: ", prec)

    accuracy_list.append(acc)
    precision_list.append(prec)

    clfs_result[name] = [acc, prec]

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  lrc
Accuracy:  0.971953578336557
Precision:  0.9504132231404959
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  svc
Accuracy:  0.971953578336557
Precision:  0.943089430894309
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  mnb
Accuracy:  0.9709864603481625
Precision:  0.9655172413793104
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  dtc
Accuracy:  0.941972920696325
Precision:  0.89
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  knc
Accuracy:  0.9294003868471954
Precision:  0.9850746268656716
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  rfc
Accuracy:  0.9748549323017408
Precision:  0.9827586206896551
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  abc
Accuracy:  0.9448742746615088
Precision:  0.9263157894736842
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  bgc
Accuracy:  0.965183752417795
Precision:  0.8805970149253731
<class 'num

c:\Users\Gangotri Mishra\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:08:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


<class 'numpy.ndarray'> <class 'numpy.ndarray'>
Classifier:  xgb
Accuracy:  0.9796905222437138
Precision:  0.968


# Maximum accuracy from Dictionary of results

*max(clfs_result, key= lambda k: clfs_result[k][0])*

here we are giving **dicitionary** "clfs_result" as input and asking get the **max**
if **we dont give key**, then it will get the maxium key based on **alphabetic order of key**.
but if we are telling them to **utilize the 0th place value from each to compare** and then give us the maximum value Key

then it will **give us maximum key based on maximum 0th place value key**

# Ways to showing desired decimal values


In [315]:
best_classifier = max(clfs_result, key= lambda k: clfs_result[k][0])

print("Best Classifier: ", best_classifier)
print("Accuracy: ", f"{clfs_result[best_classifier][0]:.2f}")
print("Precision: ", f"{clfs_result[best_classifier][1]:.2f}")

# or 

print("Accuracy: ", round(clfs_result[best_classifier][0],2))
print("Precision: ", round(clfs_result[best_classifier][1],2))

Best Classifier:  xgb
Accuracy:  0.98
Precision:  0.97
Accuracy:  0.98
Precision:  0.97


In [316]:
train_classifier.__annotations__

{}

Q4. Wrap input conversion in try/except.
Task: Write a function to_int(x) that returns an int if possible; otherwise returns None without crashing.

In [ ]:
def to_int(x):
    try:
        return(int(x))
    except ():
        return None

Safe division with else and finally.
Task: Write safe_div(a, b) that:
Returns a / b
If division by zero → return float("inf")
Always prints "done" at the end (even if error)
Use else to print "ok" only when 

In [ ]:
def safe_div(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        return float("inf")
    else:
        print("ok")
        return result
    finally:
        print("done")

NameError: name 'o' is not defined

Scenario 3 — Missing Value Strategy with else for “Happy Path”
Goal: Apply different imputation strategies and acknowledge success only when none fails.
Task
Create impute_missing(df, rules) where rules is:
rules = {"age": "median", "income": "mean", "segment": "mode"}

Rules:
If a column is missing → raise KeyError.
If numeric strategy used on non-numeric column → raise TypeError.
Use else to log "imputation complete" only if all column operations succeed.
Always print "attempted imputation" in finally.

In [396]:
df = pd.DataFrame({"age": 24, "income": 45, "segment": "google"}, index= [0])

In [ ]:
rules = {"age": "median", "income": "mean", "segment": "mode"}
df = pd.DataFrame({{"age": 24, "income": 45, "segment": "google"}})
def impute_missing(df, rules):
    try:
        

Create class Person with attributes name & age.
Subclass Employee adds salary but should still call the Person initializer.

In [15]:
class person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
class emp(person):
    def __init__(self, salary, name,age):
        super().__init__(name,age)
        self.salary = salary
        print(f"salary of {self.name} is {self.salary}")

e = emp(100,"ram",12)

Create a parent class Vehicle with method start().
Subclass Car should override start() to print "Car starts with a key".

In [18]:
class veh:
    def __init__(self):
        pass
    
    def start(self, nokey):
        self.nokey = nokey
        print(f"car start with {self.nokey}")

class car(veh):
    def __init__(self):
        pass

        # method overriding
    def start(self, start_meth):
        self.start_meth = start_meth
        print(f"car start method is {self.start_meth}")

c =car()
c.start("Big Key")

Question
Create a class BankAccount with a private balance (__balance).
Write methods to:
deposit money
get the current balance
Do NOT allow direct access to __balance.

In [24]:
class BankAccount:
    def __init__(self):
        self.__balance = 1000
        print("balance is: %s", self.__balance)
        
    
    def set_bal(self, bal):
        self.__balance = self.__balance + bal
        print("balance is: %s", self.__balance)
    
    def get_bal(self):
        return self.__balance

b = BankAccount()
b.set_bal(2000)
b.get_bal()

balance is: %s 1000
balance is: %s 3000


3000

In [2]:
a = (1,2,3)
b = (1,2,3)
a+b

(1, 2, 3, 1, 2, 3)

In [12]:
class CheckBinary:
	def __init__(self, s):
		self.s = s

	def check(self):
            x = str(self.s)
            check = 0
            for i in x:
                  print(i)
                  if i == '0' or i == '1':
                        check += 1
                  else:
                        check = -1
                        break
            if check >= 0:
                  return 'true'
            else:
                  return 'false'

In [13]:
a = CheckBinary(100110)

In [14]:
a.check()

1
0
0
1
1
0


'true'

In [5]:
import numpy as np
np.linspace(0,5,2)


array([0., 5.])

In [ ]:
np.full((4,5,3), np.arange(1,61))

ValueError: could not broadcast input array from shape (60,1) into shape (4,5,3)

In [13]:
class person:
    pass


p1 = person()
p2 = person()
print(p1==p2)

False


In [14]:
print(p1.__eq__(p2))

NotImplemented


In [27]:
class my:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return self.value == other.value 
    
class oth(my):
    def __eq__(self, other):
        return self.value == other.value*2
    
a = my(10)
c = oth(5)
a == c ## why output False

5


False

In [38]:
import pandas as pd
train_data = pd.read_csv(r'E:\ML Ops\ML_Ops-ML_Pipeline\src\data\processed\train_tfidf.csv')

In [46]:
a = train_data.head()
a

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.404514,...,0.0,0.408638,0.0,0.0,0.406145,0.0,0.0,0.000000,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,1.000000,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.262704,0.000000,...,0.0,0.000000,0.0,0.0,0.214915,0.0,0.0,0.373621,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0


In [44]:
a.shape

(5, 101)

In [63]:
a.iloc[:,-1:].values.shape

(5, 1)

In [64]:
a.iloc[:,:-1].values.shape

(5, 100)